### Arukone – Lösung Schritt für Schritt

*42. Bundeswettbewerb Informatik (2023), 1. Runde, Aufgabe 1*

Arukone sind japanische Logikrätsel. Man erhält ein Gitter, in dem jedes Feld entweder eine Zahl enthält oder leer ist.
Jede Zahl im Gitter kommt genau zwei Mal vor.

Bei der Lösung eines Arukone-Rätsels werden für jedes Paar gleicher Zahlen die entsprechenden Felder mit einem
Linienzug verbunden. Dabei gelten folgende Regeln:

- Jeder Linienzug besteht nur aus horizontalen und vertikalen Abschnitten (nicht schräg).
- In jedem Feld mit einer Zahl beginnt oder endet genau ein Linienzug.
- Jedes Feld ohne Zahl wird von genau einem Linienzug durchlaufen oder ist leer.

Auf dem Aufgabenblatt steht ein Rätsel mit Gittergröße 6 × 6 und daneben eine Lösung mit fünf farbigen Linienzügen
(`0` ist ein leeres Feld):

```
1 0 2 4 0 0
0 0 3 0 5 0
0 0 1 0 0 0
0 0 0 0 0 4
0 2 0 0 0 0
0 0 0 3 0 5
```

#### Aufgabe 1

Schreibe ein Programm, das für ein gegebenes n ≥ 4 verschiedene lösbare Arukone-Rätsel mit Gittergröße n × n erstellt,
welche mindestens n/2 verschiedene Zahlen enthalten. Unter den von deinem Programm erstellten Rätseln sollen auch welche
sein, die von dem auf den BWINF-Webseiten zur Verfügung gestellten sehr einfachen Programm (dem *Challenge-Programm* unter
[arukone.bwinf.de](https://arukone.bwinf.de/)) nicht gelöst werden.

Gib zu jedem Rätsel in deiner Dokumentation auch eine Lösung an.

#### Beispieldaten

Vorgegebene Beispiele gibt es nicht, die Aufgabe ist ja, Rätsel zu erzeugen. Das Challenge-Programm liest Dateien in diesem
Format: 1. Zeile die Gittergröße `n`, 2. Zeile die Anzahl der Zahlenpaare, danach die `n` Zeilen des Rätsels (Felder durch
Leerzeichen getrennt, leeres Feld = `0`). Das Rätsel vom Aufgabenblatt steht als `beispieldaten/arukone0.txt` im Ordner.

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Von der Lösung zum Rätsel:** Ein Rätsel zu erfinden, von dem man nicht weiß, ob es lösbar ist, ist schwer. Einfacher ist es
  rückwärts: Erst eine **Lösung** bauen (Linienzüge in ein leeres Gitter wachsen lassen), dann alles löschen bis auf die beiden
  Enden jedes Linienzugs. Das Rätsel ist dann automatisch lösbar.
- **Breitensuche (BFS):** findet in einem Gitter den **kürzesten Weg** zwischen zwei Feldern. Man läuft von Feld zu Feld in
  „Wellen“: erst alle Felder mit Abstand 1, dann mit Abstand 2 und so weiter, und merkt sich, woher man gekommen ist.
- **Greedy (gierig):** Ein Verfahren, das jede Entscheidung sofort endgültig trifft (hier: den kürzesten Weg für die Zahl 1 legen
  und nie wieder ändern). Es ist schnell, übersieht aber Lösungen, bei denen ein Weg einen Umweg machen muss, um anderen Platz
  zu lassen. Genau so ein Verfahren steckt (vermutlich) im Challenge-Programm.
- **Rückverfolgung (Backtracking):** Alle Wege durchprobieren, und wenn man in einer Sackgasse landet, den letzten Schritt
  zurücknehmen. Damit findet man immer eine Lösung, wenn es eine gibt (nur langsam).
- **Filtern durch Ausprobieren:** Zufällige Rätsel erzeugen und nur die behalten, die eine Bedingung erfüllen (hier: „Greedy
  findet keine Lösung“).

**Nützliche Funktionen und Bibliotheken**
- `random.randint`, `random.choice`, `random.shuffle`, `random.seed` (damit die Beispiele jedes Mal gleich sind)
- `collections.deque` für die Warteschlange der Breitensuche
- Listen von Listen für das Gitter, Tupel `(zeile, spalte)` für ein Feld, Dictionaries für „Zahl → ihre beiden Felder“

## Die Idee

1. **Rätsel einlesen und anzeigen** (Gitter als Liste von Listen).
2. **Prüfen, ob ein Gitter eine Lösung ist** (`pruefe_loesung`). Damit können wir jedem Ergebnis trauen.
3. Einen **Greedy-Löser** schreiben, der Zahl für Zahl den kürzesten freien Weg legt. Er soll das Challenge-Programm nachbilden.
   Wir sehen, dass er das Rätsel vom Aufgabenblatt nicht löst, obwohl es lösbar ist.
4. Einen **genauen Löser** (Rückverfolgung) schreiben, der jedes lösbare Rätsel löst. Er zeigt: Das Rätsel ist lösbar.
5. Rätsel **aus einer zufälligen Lösung erzeugen**. Sie sind lösbar, aber meistens sehr einfach.
6. Nur die Rätsel behalten, die der Greedy-Löser **nicht** löst. Das sind die gesuchten.
7. Alles zusammensetzen, ausprobieren, Beispiele dokumentieren.

## Schritt 1: Rätsel einlesen und anzeigen

Ein Rätsel ist eine Liste von Zeilen, jede Zeile eine Liste von Zahlen. Ein Feld ist ein Tupel `(zeile, spalte)`, gezählt ab 0.
Die Datei hat zwei Kopfzeilen (Größe, Anzahl der Zahlenpaare), danach kommt das Gitter.

Die Funktion `zeige` druckt ein Gitter mit `.` für leere Felder. Das brauchen wir gleich auch für Lösungen.

In [ ]:
def lies_raetsel(dateiname):
    with open(dateiname, encoding='utf-8') as f:
        zeilen = f.read().splitlines()
    n = int(zeilen[0])
    gitter = []
    for i in range(n):
        gitter.append([int(wort) for wort in zeilen[2 + i].split()])
    return gitter


def zeige(gitter):
    for zeile in gitter:
        text = ''
        for zahl in zeile:
            if zahl == 0:
                text = text + ' .'
            else:
                text = text + str(zahl).rjust(2)
        print(text)


aufgabenblatt = lies_raetsel('beispieldaten/arukone0.txt')
zeige(aufgabenblatt)

 1 . 2 4 . .
 . . 3 . 5 .
 . . 1 . . .
 . . . . . 4
 . 2 . . . .
 . . . 3 . 5


Weil wir ständig wissen müssen, wo die beiden Felder einer Zahl liegen, bauen wir ein Dictionary „Zahl → Liste ihrer beiden
Felder“. Außerdem brauchen wir für das Gitter die **Nachbarfelder** eines Feldes (oben, unten, links, rechts, soweit sie im Gitter
liegen).

In [ ]:
def finde_paare(gitter):
    paare = {}
    for z in range(len(gitter)):
        for s in range(len(gitter)):
            zahl = gitter[z][s]
            if zahl != 0:
                if zahl not in paare:
                    paare[zahl] = []
                paare[zahl].append((z, s))
    return paare


def nachbarn(n, feld):
    z, s = feld
    ergebnis = []
    for dz, ds in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
        if 0 <= z + dz < n and 0 <= s + ds < n:
            ergebnis.append((z + dz, s + ds))
    return ergebnis


def gleiche_nachbarn(gitter, feld):
    """Anzahl der Nachbarfelder, die dieselbe Zahl tragen wie feld."""
    anzahl = 0
    for nb in nachbarn(len(gitter), feld):
        if gitter[nb[0]][nb[1]] == gitter[feld[0]][feld[1]]:
            anzahl = anzahl + 1
    return anzahl


print(finde_paare(aufgabenblatt))
print(nachbarn(6, (0, 0)), nachbarn(6, (2, 3)))

{1: [(0, 0), (2, 2)], 2: [(0, 2), (4, 1)], 4: [(0, 3), (3, 5)], 3: [(1, 2), (5, 3)], 5: [(1, 4), (5, 5)]}
[(1, 0), (0, 1)] [(1, 3), (3, 3), (2, 2), (2, 4)]


## Schritt 2: Prüfen, ob ein Gitter eine Lösung ist

Eine **Lösung** schreiben wir als Gitter, in dem **jedes Feld eines Linienzugs die Nummer seines Linienzugs** trägt (leere Felder
bleiben `0`). Das Gitter für die Lösung vom Aufgabenblatt sieht also so aus wie das Rätsel, nur mit viel mehr Zahlen.

`pruefe_loesung(raetsel, loesung)` kontrolliert:
1. Die Zahlen des Rätsels stehen unverändert in der Lösung.
2. Für jede Zahl: Jedes ihrer Felder hat genau **ein** gleich beschriftetes Nachbarfeld, wenn es ein Endpunkt ist, und genau
   **zwei**, wenn es mittendrin liegt.
3. Beide Endpunkte hängen zusammen: Startet man am ersten Endpunkt und läuft über gleich beschriftete Nachbarn, erreicht man den
   zweiten, und man sieht dabei alle Felder dieser Zahl.

Aus 2 und 3 folgt, dass jede Zahl genau einen Linienzug ohne Verzweigung und ohne zusätzliche Schleifen hat. Die Prüfung
verlangt außerdem, dass ein Linienzug sich nicht selbst berührt (dann hätte ein Feld drei gleiche Nachbarn). Das schränkt die
Lösungen nicht wesentlich ein, denn solche Stellen kann man abkürzen, und aus dem beschrifteten Gitter ließe sich sonst nicht
mehr eindeutig ablesen, wie der Linienzug läuft.
Das Laufen über gleich beschriftete Nachbarn ist eine Flutung (wie bei der Breitensuche, nur ohne Entfernung).

In [ ]:
def pruefe_loesung(raetsel, loesung):
    n = len(raetsel)
    paare = finde_paare(raetsel)
    for z in range(n):
        for s in range(n):
            if raetsel[z][s] != 0 and loesung[z][s] != raetsel[z][s]:
                return False                        # Zahl des Rätsels fehlt
            if loesung[z][s] != 0 and loesung[z][s] not in paare:
                return False                        # fremde Zahl
    for zahl in paare:
        start, ziel = paare[zahl]
        felder = []
        for z in range(n):
            for s in range(n):
                if loesung[z][s] == zahl:
                    felder.append((z, s))
        for feld in felder:
            gleiche = gleiche_nachbarn(loesung, feld)
            if feld == start or feld == ziel:
                soll = 1
            else:
                soll = 2
            if gleiche != soll:
                return False
        # vom Start aus alle zusammenhängenden Felder derselben Zahl sammeln
        gesehen = [start]
        i = 0
        while i < len(gesehen):
            for nb in nachbarn(n, gesehen[i]):
                if loesung[nb[0]][nb[1]] == zahl and nb not in gesehen:
                    gesehen.append(nb)
            i = i + 1
        if ziel not in gesehen or len(gesehen) != len(felder):
            return False
    return True


# von Hand gelöst: das 3x3-Rätsel mit den Zahlen 1 und 2 aus den Lösungshinweisen (zwei verschiedene Lösungen)
kleines = [[1, 0, 2], [0, 0, 0], [1, 0, 2]]
loesung_a = [[1, 2, 2], [1, 2, 0], [1, 2, 2]]
loesung_b = [[1, 0, 2], [1, 0, 2], [1, 0, 2]]
loesung_falsch = [[1, 0, 2], [1, 0, 2], [0, 0, 2]]       # die untere 1 ist nicht verbunden
print(pruefe_loesung(kleines, loesung_a), pruefe_loesung(kleines, loesung_b), pruefe_loesung(kleines, loesung_falsch))

True True False


Das Rätsel hat also zwei verschiedene Lösungen (A läuft die 2 außen herum, B beide Zahlen geradeaus). Das ist normal: Ein Rätsel
darf mehrere Lösungen haben, es muss nur mindestens eine geben.

## Schritt 3: Ein Greedy-Löser, der das Challenge-Programm nachbildet

Die Beschreibung der Aufgabe sagt nur, das Challenge-Programm sei „sehr einfach“. Die Lösungshinweise vermuten, dass es **Zahl
für Zahl den kürzesten Weg legt und nie zurücknimmt**, also ein Greedy-Verfahren. Das passt zu den Beobachtungen dort: Vertauscht
man im Rätsel vom Aufgabenblatt die Positionen der 1en und 2en, findet das Challenge-Programm plötzlich eine Lösung. Die
Reihenfolge der Zahlen spielt also eine Rolle.

**Kürzester Weg per Breitensuche** zwischen zwei Feldern, wobei nur **freie** Felder (Wert 0) betreten werden dürfen, außer dem Ziel:

1. Ein Dictionary `vorgaenger` merkt sich zu jedem schon erreichten Feld, von wo man kam. Es dient auch als „schon besucht“.
2. Eine Warteschlange (`deque`) enthält die Felder, deren Nachbarn noch angeschaut werden müssen.
3. Nimm das vorderste Feld, schaue seine Nachbarn an; ist ein Nachbar noch nicht besucht und frei (oder das Ziel), merke dir den
   Vorgänger und hänge ihn hinten an.
4. Ist das Ziel erreicht, läuft man über die Vorgänger zurück zum Start.

Der **Greedy-Löser** ruft das für die Zahlen nacheinander auf und schreibt den gefundenen Weg ins Gitter, damit er für die
nächsten Zahlen belegt ist. Findet er für eine Zahl keinen Weg mehr, gibt er auf (`None`).

In [ ]:
from collections import deque


def kuerzester_weg(gitter, start, ziel):
    n = len(gitter)
    vorgaenger = {start: None}
    warteschlange = deque([start])
    while len(warteschlange) > 0:
        feld = warteschlange.popleft()
        if feld == ziel:
            weg = []
            while feld is not None:
                weg.append(feld)
                feld = vorgaenger[feld]
            return weg                               # vom Ziel zum Start, die Richtung ist egal
        for nb in nachbarn(n, feld):
            if nb not in vorgaenger and (nb == ziel or gitter[nb[0]][nb[1]] == 0):
                vorgaenger[nb] = feld
                warteschlange.append(nb)
    return None


def loese_greedy(raetsel, reihenfolge):
    paare = finde_paare(raetsel)
    gitter = [zeile[:] for zeile in raetsel]         # Kopie, das Rätsel selbst bleibt unverändert
    for zahl in reihenfolge:
        start, ziel = paare[zahl]
        weg = kuerzester_weg(gitter, start, ziel)
        if weg is None:
            return None
        for z, s in weg:
            gitter[z][s] = zahl
    return gitter


def checker_loest(raetsel):
    """Unser Modell des Challenge-Programms: Zahlen in der Reihenfolge 1, 2, 3, ..."""
    return loese_greedy(raetsel, sorted(finde_paare(raetsel))) is not None


print('kleines Rätsel:')
zeige(loese_greedy(kleines, [1, 2]))
print('Aufgabenblatt, Reihenfolge 1 bis 5:', loese_greedy(aufgabenblatt, [1, 2, 3, 4, 5]))

kleines Rätsel:
 1 . 2
 1 . 2
 1 . 2
Aufgabenblatt, Reihenfolge 1 bis 5: None


Das Rätsel vom Aufgabenblatt wird also vom Greedy-Löser **nicht** gelöst, genau wie vom Challenge-Programm (laut Aufgabe und
Lösungshinweisen). Der kürzeste Weg der beiden 1en blockiert den Platz, den die 2en oder die 4en brauchen. Wie sieht es mit
anderen Reihenfolgen aus? Es gibt 5! = 120 davon, wir probieren alle durch. Dazu liefert `itertools.permutations` alle
Anordnungen einer Liste.

In [ ]:
from itertools import permutations

gelungen = []
for reihenfolge in permutations([1, 2, 3, 4, 5]):
    if loese_greedy(aufgabenblatt, reihenfolge) is not None:
        gelungen.append(reihenfolge)
print(len(gelungen), 'von 120 Reihenfolgen führen zum Ziel, zum Beispiel', gelungen[0])
zeige(loese_greedy(aufgabenblatt, gelungen[0]))
print(pruefe_loesung(aufgabenblatt, loese_greedy(aufgabenblatt, gelungen[0])))

40 von 120 Reihenfolgen führen zum Ziel, zum Beispiel (2, 1, 3, 4, 5)
 1 2 2 4 4 4
 1 2 3 3 5 4
 1 2 1 3 5 4
 1 2 1 3 5 4
 1 2 1 3 5 .
 1 1 1 3 5 5
True


Mit der richtigen Reihenfolge klappt es also. Ein Rätsel, das **in keiner Reihenfolge** per Greedy lösbar ist, wäre noch
schwieriger. Das Challenge-Programm probiert vermutlich nicht alle Reihenfolgen (sonst wäre das Rätsel vom Aufgabenblatt kein
Problem für es), deshalb reicht uns das Modell „Reihenfolge 1, 2, 3, …“. Weiter unten bauen wir eine **strengere** Variante ein.

## Schritt 4: Ein genauer Löser (Rückverfolgung)

Damit wir sicher sagen können „dieses Rätsel ist lösbar“ (und damit wir auch Rätsel *ohne* bekannte Lösung prüfen können, zum
Beispiel das vom Aufgabenblatt), schreiben wir einen Löser, der **alle** Wege ausprobiert:

- Für die erste Zahl laufen wir vom ersten Feld aus Schritt für Schritt über freie Felder (jedes Feld höchstens einmal, weil wir
  es sofort mit der Zahl beschriften).
- Erreichen wir das zweite Feld der Zahl, machen wir mit der nächsten Zahl weiter. Sind alle Zahlen verbunden, ist das Rätsel gelöst.
- Kommt man nicht weiter, nehmen wir den letzten Schritt zurück (das Feld wird wieder `0`) und probieren eine andere Richtung.

Ohne Abkürzungen wäre das sehr langsam. Außerdem darf der Linienzug sich nicht selbst berühren (`sauberer_schritt`, siehe Schritt 2). Eine einfache **Abkürzung**: Bevor wir mit der nächsten Zahl weitermachen, prüfen wir,
ob **jede** noch offene Zahl überhaupt noch einen Weg hat (mit dem Breitensuche-Code von oben). Wenn nicht, lohnt sich Weitersuchen nicht.

In [ ]:
def noch_alle_erreichbar(gitter, paare, zahlen):
    for zahl in zahlen:
        start, ziel = paare[zahl]
        if kuerzester_weg(gitter, start, ziel) is None:
            return False
    return True


def sauberer_schritt(gitter, feld, nb, zahl, ziel):
    """Darf der Linienzug der Zahl von feld nach nb gehen? Das Feld nb darf außer feld (und dem Ziel) keinen weiteren
    Nachbarn mit dieser Zahl haben, sonst würde der Linienzug sich selbst berühren (und pruefe_loesung wäre nicht eindeutig)."""
    for x in nachbarn(len(gitter), nb):
        if gitter[x[0]][x[1]] == zahl and x != feld and x != ziel:
            return False
    return True


def verbinde(gitter, paare, zahlen, k, feld, ziel, zaehler):
    """Läuft von feld Richtung ziel für die Zahl zahlen[k]. Gibt True zurück, wenn alle Zahlen ab k verbunden werden konnten."""
    zaehler[0] = zaehler[0] + 1
    for nb in nachbarn(len(gitter), feld):
        if nb == ziel:
            if not sauberer_schritt(gitter, feld, nb, zahlen[k], ziel):
                continue                             # das Ziel hätte einen zweiten Nachbarn derselben Zahl
            # diese Zahl ist verbunden, jetzt die nächste
            if k + 1 == len(zahlen):
                return True
            if noch_alle_erreichbar(gitter, paare, zahlen[k + 1:]):
                start2, ziel2 = paare[zahlen[k + 1]]
                if verbinde(gitter, paare, zahlen, k + 1, start2, ziel2, zaehler):
                    return True
        elif gitter[nb[0]][nb[1]] == 0 and sauberer_schritt(gitter, feld, nb, zahlen[k], ziel):
            gitter[nb[0]][nb[1]] = zahlen[k]         # Schritt machen
            if verbinde(gitter, paare, zahlen, k, nb, ziel, zaehler):
                return True
            gitter[nb[0]][nb[1]] = 0                 # Schritt zurücknehmen
    return False


def loese_genau(raetsel):
    """Gibt die Lösung als Gitter zurück, oder None, wenn das Rätsel nicht lösbar ist (und die Zahl der Schritte)."""
    paare = finde_paare(raetsel)
    zahlen = sorted(paare)
    gitter = [zeile[:] for zeile in raetsel]
    zaehler = [0]
    start, ziel = paare[zahlen[0]]
    if verbinde(gitter, paare, zahlen, 0, start, ziel, zaehler):
        return gitter, zaehler[0]
    return None, zaehler[0]


loesung, schritte = loese_genau(aufgabenblatt)
zeige(loesung)
print('Schritte:', schritte, ' gültig:', pruefe_loesung(aufgabenblatt, loesung))

# das unlösbare 3x3-Rätsel aus den Lösungshinweisen: es lässt sich nur eine der beiden Verbindungen zeichnen
unloesbar = [[1, 2, 0], [0, 0, 0], [2, 1, 0]]
print('unlösbares Rätsel:', loese_genau(unloesbar))

 1 2 2 4 4 4
 1 2 3 3 5 4
 1 2 1 3 5 4
 1 2 1 3 5 4
 1 2 1 3 5 .
 1 1 1 3 5 5
Schritte: 30  gültig: True
unlösbares Rätsel: (None, 6)


Der genaue Löser findet für das Rätsel vom Aufgabenblatt also eine Lösung. Das Rätsel ist **lösbar**, der Greedy-Löser scheitert
trotzdem (mit der Reihenfolge 1 bis 5). Genau so ein Rätsel suchen wir. Bei dem unlösbaren Rätsel probiert er alles durch und
meldet `None`.

Die *Schrittzahl* zeigt, wie viel der Löser arbeiten muss. Bei größeren Gittern kann sie schnell sehr groß werden. Für
**unsere** Aufgabe brauchen wir ihn zum Glück kaum, denn bei selbst erzeugten Rätseln kennen wir die Lösung schon. Er ist hier
vor allem eine zweite Meinung für die Lösbarkeit.

## Schritt 5: Rätsel aus einer zufälligen Lösung erzeugen

Das Verfahren aus der Idee, in Worten:

1. Ein leeres Gitter `loesung` der Größe n × n.
2. Mische alle Felder und nimm die ersten `m` als **Startfelder** der Zahlen 1 bis m. Jede Zahl hat zuerst einen Linienzug der
   Länge 1, dessen beiden **Enden** dasselbe Feld sind.
3. Wiederhole oft (hier 3 · n²-mal): Wähle zufällig eine Zahl und eines ihrer beiden Enden. Hat das Ende ein **freies** Nachbarfeld,
   such eines davon aus, beschrifte es mit der Zahl und mache es zum neuen Ende. Der Linienzug wächst also an einem Ende um ein Feld.
   Das neue Feld darf den Linienzug nicht berühren (`sauberer_schritt`: kein weiterer Nachbar derselben Zahl).
4. Am Ende ist jede Zahl ein Linienzug zwischen ihren zwei Enden. Ist ein Linienzug nur ein Feld lang (beide Enden gleich), war er
   zu eingeklemmt, und wir beginnen von vorn.
5. Das **Rätsel** besteht nur aus den beiden Enden jeder Zahl, alle anderen Felder werden `0`.

So wächst jeder Linienzug zufällig, kreuzt aber nie einen anderen, weil er nur freie Felder betritt.

In [ ]:
import random


def zufaelliges_raetsel(n, m):
    """Gibt (raetsel, loesung) zurück: ein lösbares Rätsel mit den Zahlen 1 bis m und eine Lösung dazu."""
    while True:
        loesung = [[0] * n for z in range(n)]
        felder = [(z, s) for z in range(n) for s in range(n)]
        random.shuffle(felder)
        enden = {}                                   # Zahl -> [erstes Ende, zweites Ende]
        for zahl in range(1, m + 1):
            z, s = felder[zahl - 1]
            loesung[z][s] = zahl
            enden[zahl] = [(z, s), (z, s)]
        for versuch in range(3 * n * n):
            zahl = random.randint(1, m)
            seite = random.randint(0, 1)
            frei = []
            for nb in nachbarn(n, enden[zahl][seite]):
                if loesung[nb[0]][nb[1]] == 0 and sauberer_schritt(loesung, enden[zahl][seite], nb, zahl, None):
                    frei.append(nb)
            if len(frei) > 0:
                nb = random.choice(frei)
                loesung[nb[0]][nb[1]] = zahl
                enden[zahl][seite] = nb
        if all(enden[zahl][0] != enden[zahl][1] for zahl in enden):
            raetsel = [[0] * n for z in range(n)]
            for zahl in enden:
                for z, s in enden[zahl]:
                    raetsel[z][s] = zahl
            return raetsel, loesung


random.seed(1)
raetsel, loesung = zufaelliges_raetsel(6, 3)
zeige(raetsel)
print()
zeige(loesung)
print('gültig:', pruefe_loesung(raetsel, loesung), '  Challenge-Modell löst es:', checker_loest(raetsel))

 . . . . . .
 . . . 2 . .
 . . . . . 3
 . . . . . 2
 . . . . . 1
 . 3 1 . . .

 . 3 3 3 3 .
 3 3 2 2 3 3
 3 2 2 . . 3
 3 2 . 2 2 2
 3 2 2 2 . 1
 3 3 1 1 1 1
gültig: True   Challenge-Modell löst es: True


Die Lösung wird in jedem Fall gültig sein (das prüft `pruefe_loesung` mit). Aber ist das Rätsel schwierig? Wir erzeugen
jeweils 500 Rätsel pro Größe und zählen, bei wie vielen der Greedy-Löser **scheitert**. (Wie viele Zahlen man nimmt: mindestens
n/2, also `m = (n + 1) // 2`, das ist n/2 aufgerundet.)

In [ ]:
random.seed(2)
anteile = {}
for n in range(4, 11):
    m = (n + 1) // 2
    nicht_geloest = 0
    for i in range(500):
        raetsel, loesung = zufaelliges_raetsel(n, m)
        if not checker_loest(raetsel):
            nicht_geloest = nicht_geloest + 1
    anteile[n] = nicht_geloest
    print(f'n = {n:2d}, m = {m}: {nicht_geloest:3d} von 500 Rätseln löst der Greedy-Löser nicht ({nicht_geloest / 5:.1f} %)')

n =  4, m = 2:  61 von 500 Rätseln löst der Greedy-Löser nicht (12.2 %)
n =  5, m = 3: 120 von 500 Rätseln löst der Greedy-Löser nicht (24.0 %)
n =  6, m = 3: 145 von 500 Rätseln löst der Greedy-Löser nicht (29.0 %)
n =  7, m = 4: 200 von 500 Rätseln löst der Greedy-Löser nicht (40.0 %)
n =  8, m = 4: 173 von 500 Rätseln löst der Greedy-Löser nicht (34.6 %)
n =  9, m = 5: 252 von 500 Rätseln löst der Greedy-Löser nicht (50.4 %)
n = 10, m = 5: 233 von 500 Rätseln löst der Greedy-Löser nicht (46.6 %)


Je nach Größe sind etwa 12 bis 50 Prozent der Zufallsrätsel zu schwer für den Greedy-Löser,
bei kleinen Gittern deutlich weniger als bei großen. Die Lösungshinweise nennen als Ziel, dass im Schnitt mindestens eines von
vier erzeugten Rätseln so ein schweres ist (25 %). Für kleine Gitter erreichen wir das mit reinem Zufall nicht, und ganz
ohne Auswahl wäre in der Aufgabe auch nicht gesichert, dass **überhaupt** eines dabei ist.

## Schritt 6: Nur die schwierigen behalten

Die Lösung ist ganz einfach: **So lange Rätsel erzeugen, bis der Greedy-Löser eines nicht löst.** Das Rätsel ist lösbar
(wir haben ja eine Lösung dazu), und das Challenge-Programm (bzw. unser Modell davon) scheitert. Die Zahl der nötigen Versuche
liefern wir mit, damit man sieht, wie lange das dauert (kein Zeitmessen, das wäre nicht reproduzierbar).

Der Parameter `streng` verschärft die Bedingung: Dann darf der Greedy-Löser **auch in umgekehrter Reihenfolge** (m, …, 2, 1) und
in 10 zufälligen Reihenfolgen nichts finden. Das passt besser zu der Vermutung, dass das Challenge-Programm nicht nur nach der
Nummer vorgeht.

In [ ]:
def zu_schwer_fuer_greedy(raetsel, streng):
    zahlen = sorted(finde_paare(raetsel))
    if loese_greedy(raetsel, zahlen) is not None:
        return False
    if streng:
        if loese_greedy(raetsel, zahlen[::-1]) is not None:
            return False
        for i in range(10):
            random.shuffle(zahlen)
            if loese_greedy(raetsel, zahlen) is not None:
                return False
    return True


def erzeuge_raetsel(n, streng=False):
    """Gibt (raetsel, loesung, versuche) zurück. Das Rätsel hat (n + 1) // 2 oder eine Zahl mehr verschiedene Zahlen."""
    m = (n + 1) // 2 + random.randint(0, 1)
    versuche = 0
    while True:
        versuche = versuche + 1
        raetsel, loesung = zufaelliges_raetsel(n, m)
        if zu_schwer_fuer_greedy(raetsel, streng):
            return raetsel, loesung, versuche


random.seed(3)
raetsel, loesung, versuche = erzeuge_raetsel(6)
zeige(raetsel)
print('Versuche:', versuche)
zeige(loesung)
print('Lösung gültig:', pruefe_loesung(raetsel, loesung), '  Greedy (1, 2, 3, ...) löst es:', checker_loest(raetsel))

 . . . 2 . 3
 . . . . . .
 . . . . . 3
 1 . . . . .
 . . . 2 . 1
 . . . . . .
Versuche: 4
 2 2 2 2 3 3
 2 . . 3 3 .
 2 2 . 3 . 3
 1 2 . 3 3 3
 1 2 2 2 1 1
 1 1 1 1 1 .
Lösung gültig: True   Greedy (1, 2, 3, ...) löst es: False


## Schritt 7: Beispiele für verschiedene Größen

Wir erzeugen Rätsel für n = 4 bis 10, für jede Größe das schwere (nach dem Challenge-Modell) und prüfen für jedes:

- die Lösung ist gültig (`pruefe_loesung`),
- der Greedy-Löser scheitert,
- es gibt mindestens n/2 verschiedene Zahlen,
- alle Rätsel sind **verschieden** (wir sammeln sie in einer Menge; Listen kann man nicht in eine Menge legen, deshalb wandeln wir
  jedes Gitter in ein Tupel von Tupeln um).

Danach schreiben wir die Rätsel im Dateiformat des Challenge-Programms, damit man sie dort eingeben kann.

In [ ]:
def als_datei_text(raetsel):
    m = len(finde_paare(raetsel))
    zeilen = [str(len(raetsel)), str(m)]
    for zeile in raetsel:
        zeilen.append(' '.join(str(zahl) for zahl in zeile))
    return '\n'.join(zeilen)


random.seed(4)
alle = set()
beispiele = {}
for n in range(4, 11):
    raetsel, loesung, versuche = erzeuge_raetsel(n)
    m = len(finde_paare(raetsel))
    ok = pruefe_loesung(raetsel, loesung) and not checker_loest(raetsel) and 2 * m >= n
    alle.add(tuple(tuple(zeile) for zeile in raetsel))
    beispiele[n] = (raetsel, loesung)
    print(f'n = {n:2d}: {m} Zahlen, {versuche:4d} Versuche, alles in Ordnung: {ok}')
print('verschiedene Rätsel:', len(alle))

n =  4: 2 Zahlen,    5 Versuche, alles in Ordnung: True
n =  5: 4 Zahlen,    2 Versuche, alles in Ordnung: True
n =  6: 4 Zahlen,    2 Versuche, alles in Ordnung: True
n =  7: 5 Zahlen,    1 Versuche, alles in Ordnung: True
n =  8: 4 Zahlen,    2 Versuche, alles in Ordnung: True
n =  9: 5 Zahlen,    4 Versuche, alles in Ordnung: True
n = 10: 5 Zahlen,    3 Versuche, alles in Ordnung: True
verschiedene Rätsel: 7


Drei davon im Einzelnen, mit Lösung. Die Dateitexte zum Eintippen oder Hochladen beim Challenge-Programm stehen jeweils darüber.
Wir haben dieses Programm nicht eingebaut; deshalb ist unsere Aussage „der Checker findet keine Lösung“ eine **Annahme** aus dem
Greedy-Modell. Probiere die Dateien am besten selbst auf [arukone.bwinf.de](https://arukone.bwinf.de/) aus.

In [ ]:
for n in [5, 7, 9]:
    raetsel, loesung = beispiele[n]
    print('Datei für den Checker:')
    print(als_datei_text(raetsel))
    print()
    print('Lösung:')
    zeige(loesung)
    print('-' * 30)

Datei für den Checker:
5
4
0 0 0 3 2
0 0 0 1 0
0 0 1 4 0
0 0 0 0 0
3 4 2 0 0

Lösung:
 . 3 3 3 2
 . 3 1 1 2
 3 3 1 4 2
 3 4 4 4 2
 3 4 2 2 2
------------------------------
Datei für den Checker:
7
5
3 0 0 0 0 0 0
0 0 5 0 0 0 0
3 0 1 0 0 0 0
2 0 0 0 0 0 0
0 4 0 0 0 0 1
0 0 0 0 0 0 5
0 0 4 2 0 0 0

Lösung:
 3 1 1 1 1 . .
 3 1 5 5 1 1 1
 3 1 1 5 5 5 1
 2 2 2 . . 5 1
 4 4 2 . 5 5 1
 4 . 2 2 5 . 5
 4 4 4 2 5 5 5
------------------------------
Datei für den Checker:
9
5
0 4 0 4 0 0 0 0 0
0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0
0 0 0 3 0 0 0 0 3
0 0 5 0 0 0 0 0 2
0 0 0 0 0 2 0 0 0
1 0 0 0 0 0 0 0 1
0 0 5 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0

Lösung:
 4 4 . 4 3 3 3 3 .
 4 . 4 4 3 . . 3 3
 4 4 4 . 3 3 3 . 3
 5 5 5 3 . . 3 . 3
 5 . 5 3 3 3 3 . 2
 5 5 . 2 2 2 . 2 2
 1 5 . 2 . . . 2 1
 1 5 5 2 2 2 2 2 1
 1 1 1 1 1 1 1 1 1
------------------------------


### Strenge Rätsel

Mit `streng=True` suchen wir Rätsel, die auch bei anderen Reihenfolgen scheitern. Das dauert länger, weil mehr Rätsel
aussortiert werden. Bei n = 4 mit zwei Zahlen haben wir kein solches Rätsel gefunden. Wir probieren n = 6 bis 8 und zählen
mit `permutations` nach, bei wie vielen Reihenfolgen der Greedy-Löser doch noch Erfolg hat (wir haben ja nur 12 Reihenfolgen getestet):

In [ ]:
random.seed(5)
for n in [6, 7, 8]:
    raetsel, loesung, versuche = erzeuge_raetsel(n, streng=True)
    zahlen = sorted(finde_paare(raetsel))
    anzahl_ok = 0
    for reihenfolge in permutations(zahlen):
        if loese_greedy(raetsel, reihenfolge) is not None:
            anzahl_ok = anzahl_ok + 1
    print(f'n = {n}: {versuche} Versuche, Lösung gültig: {pruefe_loesung(raetsel, loesung)}, ' +
          f'von {len(list(permutations(zahlen)))} Reihenfolgen führen {anzahl_ok} zum Ziel')
    if n == 6:
        zeige(raetsel)

n = 6: 3 Versuche, Lösung gültig: True, von 24 Reihenfolgen führen 3 zum Ziel
 2 . 3 . . .
 4 . . . . .
 . 1 . 2 . .
 . . . . . 3
 . . . 4 . .
 . . . . . 1
n = 7: 2 Versuche, Lösung gültig: True, von 120 Reihenfolgen führen 20 zum Ziel
n = 8: 11 Versuche, Lösung gültig: True, von 24 Reihenfolgen führen 0 zum Ziel


## Für Neugierige

- **Muster statt Zufall:** Die Lösungshinweise schlagen auch vor, ein kleines Rätsel zu verwenden, das der Checker sicher nicht
  löst, und es mit zufälligen Teilen zu einem größeren zu ergänzen (Beispiel 8 × 8 dort). Man muss darauf achten, dass die Ergänzung dem
  Greedy-Verfahren keine neuen Umwege eröffnet. Unser Zufallsverfahren braucht das nicht.
- **Echte Schwierigkeit:** Ob ein Rätsel *für Menschen* schwer ist, sagt unser Greedy-Löser nicht. Ein Maß könnte die Zahl der
  Rückverfolgungsschritte von `loese_genau` sein. Auch die Zahl der **Lösungen** ist wichtig: Gute Arukone haben genau eine.
- **Nachbessern:** Man kann ein Rätsel verändern (zwei Zahlen vertauschen, ein Ende verschieben) und prüfen, ob es lösbar bleibt und
  schwerer wird (*Hill-Climbing*).

## Zum Schluss: Die Lösbarkeit mit MiniZinc prüfen

Ob ein Rätsel lösbar ist, kann man auch als **Constraint-Problem** schreiben. Die Idee: Gib **jedem Feld eine Farbe** `0` bis `m`
(`0` = leer, `i` = Linienzug `i`). Dann beschreiben wir die Regeln der Aufgabe als Bedingungen auf den Nachbarfeldern:

- Ein Feld mit einer Zahl hat diese Farbe, und **genau ein** Nachbarfeld hat dieselbe Farbe (hier beginnt/endet der Linienzug).
- Ein nicht leeres Feld ohne Zahl hat **genau zwei** Nachbarfelder derselben Farbe (hier läuft der Linienzug durch).

Das reicht für Lösbarkeit. Es kann passieren, dass der Solver nebenbei einen **geschlossenen Ring** aus Feldern einer Farbe
irgendwo ins Gitter malt (jedes Ringfeld hat zwei gleiche Nachbarn). Die beiden Enden einer Zahl hängen trotzdem zusammen, denn von
einem Ende aus kommt man nur zu einem Feld mit genau einem gleichen Nachbarn, und das kann nur das andere Ende sein. Den Ring
könnte man einfach weglassen. Ob es eine Lösung gibt, sagt das Modell also richtig.

Die Modelle sind **kein Python**: Kopiere sie in die MiniZinc IDE (oder in den
[MiniZinc Playground](https://play.minizinc.org/)). Wir haben den Solver **Chuffed** benutzt (er lernt aus Sackgassen und ist
hier viel schneller als Gecode).

Das Modell (`arukone.mzn`):

```minizinc
int: n;
int: m;
array[1..n, 1..n] of int: gitter;                    % das Rätsel, 0 = leer
array[1..n, 1..n] of var 0..m: farbe;                % die Lösung: Farbe jedes Feldes

% Zahlenfelder haben ihre Zahl als Farbe
constraint forall(z in 1..n, s in 1..n where gitter[z,s] > 0)(farbe[z,s] = gitter[z,s]);

% Zahlenfeld: genau 1 Nachbar mit gleicher Farbe
constraint forall(z in 1..n, s in 1..n where gitter[z,s] > 0)(
  sum(r in [-1, 1])(
      bool2int(z + r in 1..n /\ farbe[max(1, min(n, z + r)), s] = farbe[z, s])
    + bool2int(s + r in 1..n /\ farbe[z, max(1, min(n, s + r))] = farbe[z, s])) = 1);

% Feld ohne Zahl: leer (Farbe 0) oder genau 2 Nachbarn mit gleicher Farbe
constraint forall(z in 1..n, s in 1..n where gitter[z,s] = 0)(
  farbe[z,s] = 0 \/
  sum(r in [-1, 1])(
      bool2int(z + r in 1..n /\ farbe[max(1, min(n, z + r)), s] = farbe[z, s])
    + bool2int(s + r in 1..n /\ farbe[z, max(1, min(n, s + r))] = farbe[z, s])) = 2);

solve satisfy;
output [ show(farbe[z,s]) ++ (if s = n then "\n" else " " endif) | z in 1..n, s in 1..n ];
```

Ein Wort dazu: `r in [-1, 1]` läuft über die beiden Richtungen. `max(1, min(n, …))` schützt den Index, falls der Nachbar
außerhalb liegt (dort ist die Bedingung `z + r in 1..n` ohnehin falsch, der Nachbar zählt also nicht mit).
Die Datendatei erzeugt diese Python-Zelle aus einem Rätsel (Rätsel als Liste von Listen → `.dzn`-Text):

In [ ]:
def als_dzn(raetsel):
    n = len(raetsel)
    m = len(finde_paare(raetsel))
    felder = []
    for zeile in raetsel:
        for zahl in zeile:
            felder.append(str(zahl))
    return ('n = ' + str(n) + ';\nm = ' + str(m) + ';\ngitter = array2d(1..' + str(n) + ', 1..' + str(n) +
            ', [' + ', '.join(felder) + ']);')


print(als_dzn(aufgabenblatt))

n = 6;
m = 5;
gitter = array2d(1..6, 1..6, [1, 0, 2, 4, 0, 0, 0, 0, 3, 0, 5, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, 5]);


Für das Rätsel vom Aufgabenblatt liefert Chuffed (Farben der Felder, `0` = leer):

**Ausgabe:**

```
1 2 2 4 4 4
1 2 3 3 5 4
1 2 1 3 5 4
1 2 1 3 5 4
1 2 1 3 5 0
1 1 1 3 5 5
----------
```

Das ist dieselbe Lösung wie beim Python-Löser. Beim unlösbaren 3 × 3-Rätsel
`[[1, 2, 0], [0, 0, 0], [2, 1, 0]]` meldet der Solver sofort, dass es keine Lösung gibt:

**Ausgabe:**

```
=====UNSATISFIABLE=====
```

**Grenzen:** Für unsere erzeugten Rätsel bis 10 × 10 antwortet Chuffed praktisch sofort. Gecode brauchte bei einzelnen Rätseln
ab 8 × 8 dagegen mehr als zehn Sekunden, auch Chuffed bei einzelnen 12 × 12. Das Modell ist also gut, um die Lösbarkeit zu
prüfen. Zum *Erzeugen* kommen wir mit der Rückwärts-Methode aber viel schneller zum Ziel.